# Lab 14 — Data Poisoning Robustness

*AI Cybersecurity Masterclass · Lab 14 · AI60secLab*

**Research question:** How quickly does detection performance degrade when a small fraction of training data is corrupted?

We study two controlled offline conditions:
1. random binary label flips;
2. bounded numeric feature corruption.

The official test set remains clean.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import logistic_pipeline
from src.evaluation import binary_metrics
from src.poisoning import (
    random_label_flip,
    corrupt_numeric_training_rows,
    poisoning_summary,
)
from src.robustness import feature_bounds

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [ ]:
label_rows = []

for fraction in [0.00, 0.01, 0.03, 0.05, 0.10]:
    poisoned_y = random_label_flip(
        y_train,
        fraction=fraction,
        random_state=42,
    )

    model = logistic_pipeline(X_train)
    model.fit(X_train, poisoned_y)

    pred = model.predict(X_test)
    score = model.predict_proba(X_test)[:, 1]
    metrics = binary_metrics(y_test, pred, score)
    summary = poisoning_summary(y_train, poisoned_y)

    label_rows.append({
        "requested_fraction": fraction,
        "actual_changed_fraction": summary["changed_fraction"],
        "precision": metrics.precision,
        "recall": metrics.recall,
        "f1": metrics.f1,
        "false_negative_rate": metrics.false_negative_rate,
    })

label_results = pd.DataFrame(label_rows)
display(label_results.round(4))

In [ ]:
numeric_columns = [
    c for c in [
        "dur", "sbytes", "dbytes", "spkts",
        "dpkts", "rate", "sload", "dload"
    ]
    if c in X_train.columns
]

bounds = feature_bounds(X_train, numeric_columns)
feature_rows = []

for fraction in [0.00, 0.01, 0.03, 0.05, 0.10]:
    poisoned_X = corrupt_numeric_training_rows(
        X_train,
        fraction=fraction,
        columns=numeric_columns,
        noise_fraction=0.30,
        bounds=bounds,
        random_state=42,
    )

    model = logistic_pipeline(poisoned_X)
    model.fit(poisoned_X, y_train)

    pred = model.predict(X_test)
    score = model.predict_proba(X_test)[:, 1]
    metrics = binary_metrics(y_test, pred, score)

    feature_rows.append({
        "corrupted_row_fraction": fraction,
        "precision": metrics.precision,
        "recall": metrics.recall,
        "f1": metrics.f1,
        "false_negative_rate": metrics.false_negative_rate,
    })

feature_results = pd.DataFrame(feature_rows)
display(feature_results.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    label_results["requested_fraction"],
    label_results["f1"],
    marker="o",
    label="Label corruption",
)
ax.plot(
    feature_results["corrupted_row_fraction"],
    feature_results["f1"],
    marker="o",
    label="Feature corruption",
)
ax.set_xlabel("Corrupted fraction")
ax.set_ylabel("F1")
ax.set_title("Training-data corruption sensitivity")
ax.legend()
plt.show()

## Interpretation

Which corruption type damaged performance faster? How would provenance checks, validation rules, and trusted training pipelines reduce poisoning risk?

**Takeaway:** training data is part of the ML system's security boundary.